In [1]:
#display all the results not only the last one
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import torch as tc
from numba import cuda
import numpy as np
import time

def sample_strengths(N, stre_out_method = 'correlated'):
    """
    Sample strengths from a log-normal with fixed mu and scale
    if stre_out_method = 'correlated' a positive correlation will be set among the prop_out, prop_in
    """
    from numpy import random as npr

    # sample the strengths in from the log-normal distribution
    mu_in, scale_in = 9.145050968796784, 3.0501786627011396
    npr.seed(0)
    logn_sample = lambda mu, scale: npr.lognormal(mean=mu, sigma=scale, size=N)
    prop_in = logn_sample(mu_in, scale_in)

    # sample the strengths out from the log-normal distribution
    mu_out, scale_out = 9.753643239750687, 3.2045983690423174
    if stre_out_method == 'uncorrelated':
        prop_out = logn_sample(mu_out, scale_out)
    elif stre_out_method == 'correlated':
        mu_eps, scale_eps = mu_out - mu_in, np.sqrt(scale_out**2 - scale_in**2)
        eps = logn_sample(mu_eps, scale_eps)
        prop_out = prop_in * eps


    return prop_out, prop_in

In [3]:
dev = "cuda"
p = tc.randn(1,2).to(dev)
tc.rand_like(p).device
(p * 2).device

device(type='cuda', index=0)

device(type='cuda', index=0)

In [4]:
def tc_p_ij(d, x_i, y_j, z_ij):
    """Compute the probability of connection between node i and j."""

    tmp = d * x_i * y_j
    return -tc.expm1(-tmp)

def tc_prop_dyad(i, j):
    return 1  # dummy dyad

def block_parallel_sample_tc(param, prop_out, prop_in, selfloops, unsampled_vI, chunk_row_size=100, seed = 0):
    """
    Efficiently sample edges in chunks to avoid GPU memory overflow.
    """
    N = prop_out.shape[0]
    device = "cuda"

    # Seed once at the beginning
    if device == 'cuda':
        tc.cuda.manual_seed(seed)
    else:
        tc.manual_seed(seed)

    # Prepare output lists to collect tensors directly on GPU
    all_rows = []
    all_cols = []

    # Process in chunks to avoid memory issues
    for start in range(0, N, chunk_row_size):
        end = min(start + chunk_row_size, N)
        idx_i = tc.arange(start, end, device=device)
        idx_j = tc.arange(N, device=device)


        # get the flat list of all the pairs
        grid_i = tc.repeat_interleave(idx_i, len(idx_j))
        grid_j = idx_j.repeat(len(idx_i))

        # Mask some pairs
        mask = tc.ones_like(grid_i, dtype=tc.bool)

        # if False, when grid_i == grid_j --> mask = False
        if not selfloops:
            mask &= (grid_i != grid_j)
        # Set mask = False when both unsampled_vI are True
        if unsampled_vI.any():
            mask &= ~(unsampled_vI[grid_i] & unsampled_vI[grid_j])

        # Retain the pairs to sample
        grid_i = grid_i[mask]
        grid_j = grid_j[mask]

        if grid_i.size == 0:
            continue # No pairs to process in this chunk

        # Compute probabilities over the unmasked pairs
        x_i, y_j = prop_out[grid_i], prop_in[grid_j]
        p = tc_p_ij(param, x_i, y_j, tc_prop_dyad(grid_i, grid_j))

        # Sample edges
        rand = tc.rand_like(p)
        selected_idx = tc.where(rand < p)
        grid_i = grid_i[selected_idx]
        grid_j = grid_j[selected_idx]

        # Append tensors directly on the GPU
        all_rows.append(grid_i)
        all_cols.append(grid_j)

    # Concatenate all results on the GPU, then transfer to CPU once
    if all_rows: # Check if list is not empty
        rows = tc.cat(all_rows)
        cols = tc.cat(all_cols)

        rows = rows.cpu().numpy()
        cols = cols.cpu().numpy()
    else:
        # No edges sampled, return empty arrays
        rows = tc.empty(0, dtype=tc.int64).numpy()
        cols = tc.empty(0, dtype=tc.int64).numpy()


    return rows, cols

In [12]:
from math import expm1 

@cuda.jit
def nb_p_ij_kernel(d, x_i, y_j):
    """Compute the probability of connection between node i and j (device function)."""
    tmp = d * x_i * y_j
    return -expm1(-tmp)

@cuda.jit
def sample_chunk(
    param, prop_out, prop_in,
    grid_i_chunk, grid_j_chunk, mask_chunk, p_chunk, rand_chunk,
    rows_out, cols_out, count_out,
):
    """
    CUDA kernel for sampling edges within a chunk.
    This kernel is called for each pair (i, j) in the chunk.
    """
    k = cuda.grid(1) # Global thread index

    N_chunk_pairs = grid_i_chunk.shape[0]

    if k < N_chunk_pairs:
        i = grid_i_chunk[k]
        j = grid_j_chunk[k]

        # Use the pre-computed mask
        if not mask_chunk[k]:   
            return

        # Compute probability
        x_i = prop_out[i]
        y_j = prop_in[j]
        p = nb_p_ij_kernel(param, x_i, y_j)
        p_chunk[k] = p # Store p for debugging/verification if needed

        # Generate random number for this thread
        # Note: Numba's CUDA random number generation is not as high-quality
        # as PyTorch's for complex scientific simulations, but sufficient for a quick benchmark.
        # For production, consider curand.
        # A simple XORShift or similar can be implemented. For this example, we will just use a pseudo-random
        # based on thread ID and seed, which is very basic. For a real comparison, a proper
        # GPU RNG (like cuRAND via Numba) would be needed.
        # For simplicity, we will assume rand_chunk is pre-filled with random numbers on host.
        # In a real kernel, you would generate random numbers on the device using proper CUDA RNG.
        # For a fair comparison with PyTorch's tc.rand_like, we will pass pre-generated random numbers.

        # Sample edge
        if rand_chunk[k] < p:
            # Atomically increment a counter and store the indices
            idx = cuda.atomic.add(count_out, 0, 1)
            rows_out[idx] = i
            cols_out[idx] = j

def block_parallel_sample_numba(param_d, prop_out_d, prop_in_d, selfloops, unsampled_vI, chunk_row_size=100, 
    # Calculate blocks and threads per block
    threads_per_block = 256, seed=0):
    """
    Efficiently sample edges in chunks using Numba-CUDA.
    """
    N = prop_out_d.shape[0]

    all_rows = []
    all_cols = []

    for start in range(0, N, chunk_row_size):
        end = min(start + chunk_row_size, N)
        idx_i = np.arange(start, end, dtype=np.int64)
        idx_j = np.arange(N, dtype=np.int64)

        # Generate all pairs for the chunk
        grid_i = np.repeat(idx_i, len(idx_j))
        grid_j = np.tile(idx_j, len(idx_i))

        # Masking
        mask = np.ones_like(grid_i, dtype=np.bool_)
        if not selfloops:
            mask &= (grid_i != grid_j)
        if unsampled_vI.any(): # Only apply if there are any unsampled nodes
            mask &= ~(unsampled_vI[grid_i] & unsampled_vI[grid_j])

        # Filter the pairs based on the mask
        grid_i = grid_i[mask]
        grid_j = grid_j[mask]

        if grid_i.size == 0:
            continue # No pairs to process in this chunk

        # Transfer filtered pairs and mask to device
        grid_i_d = cuda.to_device(grid_i)
        grid_j_d = cuda.to_device(grid_j)
        mask_d = cuda.to_device(mask) # Numba kernel uses this pre-masked list directly

        # Allocate device memory for probabilities and random numbers
        num_pairs = grid_i.shape[0]
        p_d = cuda.device_array(num_pairs, dtype=np.float32)

        # Generate random numbers on host and transfer to device for fair comparison with PyTorch's `tc.rand_like`
        # For better performance and true GPU randomness, `numba_cuda_rng` or similar should be used.
        np.random.seed(seed + start) # Vary seed per chunk for pseudo-randomness
        rand = np.random.rand(num_pairs).astype(np.float32)
        rand_d = cuda.to_device(rand)

        # Allocate output arrays on device. Max possible edges is num_pairs.
        rows_d = cuda.device_array(num_pairs, dtype=np.int64)
        cols_d = cuda.device_array(num_pairs, dtype=np.int64)
        count_d = cuda.to_device(np.array([0], dtype=np.int64)) # Atomic counter for sampled edges

        # Calculate blocks per grid
        blocks_per_grid = (num_pairs + (threads_per_block - 1)) // threads_per_block

        # Launch the kernel
        sample_chunk[blocks_per_grid, threads_per_block](
            param_d, prop_out_d, prop_in_d,
            grid_i_d, grid_j_d, mask_d, p_d, 
            rand_d, rows_d, cols_d, count_d,
        )
        cuda.synchronize() # Wait for kernel to finish

        # Retrieve results
        num_sampled_edges = count_d.copy_to_host()[0]
        if num_sampled_edges > 0:
            all_rows.append(rows_d[:num_sampled_edges].copy_to_host())
            all_cols.append(cols_d[:num_sampled_edges].copy_to_host())

    if all_rows:
        rows = np.concatenate(all_rows)
        cols = np.concatenate(all_cols)
    else:
        rows = np.empty(0, dtype=np.int64)
        cols = np.empty(0, dtype=np.int64)

    return rows, cols

In [15]:
# Ensure CUDA is available
if not tc.cuda.is_available():
    print("CUDA is not available. Please enable CUDA for PyTorch and Numba.")
    exit()

# Parameters for testing
N_nodes = 10000  # Number of nodes
chunk_size = 300
param = 0.1
selfloops_val = False
seed = 42
device = "cuda"

# Generate synthetic data
prop_out, prop_in = sample_strengths(N_nodes)
# prop_out = np.random.rand(N_nodes).astype(np.float32) * 2
# prop_in = np.random.rand(N_nodes).astype(np.float32) * 2
unsampled_vI = np.zeros_like(prop_out, dtype=np.bool)

param_d = cuda.to_device(np.array([param], dtype=np.float32))
prop_out_d = cuda.to_device(prop_out.astype(np.float32))
prop_in_d = cuda.to_device(prop_in.astype(np.float32))
# unsampled_vI = cuda.to_device(unsampled_vI)

cuda.synchronize()
start_time_nb = time.perf_counter()
rows_nb, cols_nb = block_parallel_sample_numba(
    param, prop_out, prop_in, selfloops_val, unsampled_vI, chunk_size, threads_per_block = 1024, seed = seed
)

In [11]:
# Ensure CUDA is available
if not tc.cuda.is_available():
    print("CUDA is not available. Please enable CUDA for PyTorch and Numba.")
    exit()

# Parameters for testing
N_nodes = 10000  # Number of nodes
chunk_size = 300
param = 0.1
selfloops_val = False
seed = 42
device = "cuda"

# Generate synthetic data
prop_out, prop_in = sample_strengths(N_nodes)
# prop_out = np.random.rand(N_nodes).astype(np.float32) * 2
# prop_in = np.random.rand(N_nodes).astype(np.float32) * 2
unsampled_vI = np.zeros_like(prop_out, dtype=np.bool)

print(f"Benchmarking with N_nodes={N_nodes}, chunk_size={chunk_size}")

# Run PyTorch version
# convert the arrays to PyTorch and send them into GPUs
param = tc.tensor(param, device=device)
prop_out = tc.from_numpy(prop_out).cuda()
prop_in = tc.from_numpy(prop_in).cuda()
unsampled_vI = tc.from_numpy(unsampled_vI).cuda()

tc.cuda.synchronize()
start_time_tc = time.perf_counter()
rows_tc, cols_tc = block_parallel_sample_tc(
    param, prop_out, prop_in, selfloops_val, unsampled_vI, chunk_size, seed
)
tc.cuda.synchronize()
end_time_tc = time.perf_counter()
time_tc = end_time_tc - start_time_tc
print(f"PyTorch execution time: {time_tc:.6f} seconds")
print(f"PyTorch sampled edges: {len(rows_tc)}")

# Run Numba-CUDA version
# convert the arrays to PyTorch and send them into GPUs
# Transfer parameters to GPU
param_d = cuda.to_device(np.array([param], dtype=np.float32))
prop_out_d = cuda.to_device(prop_out.astype(np.float32))
prop_in_d = cuda.to_device(prop_in.astype(np.float32))
unsampled_vI = cuda.to_device(unsampled_vI)

cuda.synchronize()
start_time_nb = time.perf_counter()
rows_nb, cols_nb = block_parallel_sample_numba(
    param, prop_out, prop_in, selfloops_val, unsampled_vI, chunk_size, threads_per_block = 512, seed = seed
)
cuda.synchronize()
end_time_nb = time.perf_counter()
time_nb = end_time_nb - start_time_nb
print(f"Numba-CUDA execution time: {time_nb:.6f} seconds")
print(f"Numba-CUDA sampled edges: {len(rows_nb)}")

# Verification (optional: check if results are identical, though random sampling makes this hard)
# For a perfect match, the random seeds and generation process would need to be identical.
# We can check if the number of sampled edges is roughly similar.
# print(f"Are sampled edge counts similar? {abs(len(rows_tc) - len(rows_nb)) < max(len(rows_tc), len(rows_nb)) * 0.05}")

if time_nb < time_tc:
    print(f"\nNumba-CUDA is faster by {time_tc / time_nb:.2f}x")
elif time_tc < time_nb:
    print(f"\nPyTorch is faster by {time_nb / time_tc:.2f}x")
else:
    print("\nBoth implementations have similar performance.")

Benchmarking with N_nodes=10000, chunk_size=300
PyTorch execution time: 1.571086 seconds
PyTorch sampled edges: 99985318


TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.